# Lovable Backtest — Laboratorio Python

Cuaderno Colab que reproduce las 6 estrategias del dashboard TypeScript.
Usa Colab (o Kaggle) para no cargar tu PC. Ver `README.md` para el modelo
de trabajo (Modelo B: Python = lab, Dashboard = producción).

## 1. Setup — instalar deps y cargar la librería

In [ ]:
# En Colab: sube este directorio o clona el repo.
# Si estás corriendo local: `pip install -r requirements.txt` (una vez).
%pip install -q numpy pandas joblib matplotlib

import os, sys
# Asegura que Python encuentra lovable_backtest.py junto al notebook
HERE = os.path.dirname(os.path.abspath('__file__')) if '__file__' in globals() else os.getcwd()
if HERE not in sys.path: sys.path.insert(0, HERE)

import lovable_backtest as lb
import pandas as pd, numpy as np, json
print('Estrategias disponibles:', list(lb.STRATEGIES.keys()))

## 2. Cargar CSV

Sube tus CSV de MT5 al notebook (mismos que usas en el dashboard). Se aceptan
dos formatos: `YYYY.MM.DD HH:MM,O,H,L,C,V` (script MT5) o `MM/DD/YYYY HH:MM,O,H,L,C,V` (Investing/Dukascopy).

Si solo tienes M1, `load_bars` agrega M5/M15/H1/H4 automáticamente.

In [ ]:
# Ajusta rutas: puede ser solo {'M1': 'xauusd_m1.csv'} o el set completo.
csv_files = {
    'M1':  'data/xauusd_m1.csv',
    # 'M5':  'data/xauusd_m5.csv',
    # 'M15': 'data/xauusd_m15.csv',
    # 'H1':  'data/xauusd_h1.csv',
    # 'H4':  'data/xauusd_h4.csv',
}
bars = lb.load_bars(csv_files, build_missing=True)
for tf, df in bars.items():
    print(f'{tf}: {len(df):>7} velas · desde {pd.to_datetime(df.time.min(), unit="s")} hasta {pd.to_datetime(df.time.max(), unit="s")}')

## 3. Backtest simple (sanity check)

In [ ]:
engine = 'fibo_scalping'  # prueba: fibo_scalping, gold_scalping, ema_cross_m1, straddle_breakout, smc_london, ny_continuation
res = lb.run_backtest_bars(bars, engine)
print(json.dumps(res['metrics'], indent=2, default=str))
df_trades = lb.trades_to_df(res['trades'])
df_trades.head()

## 4. Grid optimizer (paralelo con joblib)

El grid vive en `strategies_spec.json`. Puedes editarlo o pasar uno propio.

In [ ]:
spec_path = os.path.join(HERE, 'strategies_spec.json')
spec = json.load(open(spec_path))
grid = spec['engines'][engine]['grid']
print('Grid:', grid)
df_grid = lb.grid_search(bars, engine, grid, min_trades=10, n_jobs=-1)
df_grid.head(10)

## 5. Walk-forward (train N meses / test M meses)

Rolling window: optimiza en train y evalúa OOS en test. La media de las
métricas OOS es lo que realmente importa (no la mejor combinación in-sample).

In [ ]:
df_wf = lb.walk_forward(bars, engine, grid, train_months=3, test_months=1, min_trades=10, n_jobs=-1)
print('Ventanas OOS:', len(df_wf))
print('Winrate OOS media :', df_wf['oos_winrate'].mean() if len(df_wf) else 'n/a')
print('Expectancy OOS media:', df_wf['oos_avg_r'].mean() if len(df_wf) else 'n/a')
df_wf

## 6. Export → best_params.json

Consumido por el dashboard TS y (más adelante) por el EA MT5 vía la tabla `mt5_signals`.

In [ ]:
# Ejemplo: optimizar TODAS las estrategias y exportar el mejor de cada una.
results = {}
for key, engine_def in lb.STRATEGIES.items():
    g = spec['engines'][key]['grid']
    try:
        df = lb.grid_search(bars, key, g, min_trades=10, n_jobs=-1)
        if df.empty: continue
        top = df.iloc[0]
        params = {k: (int(top[k]) if isinstance(top[k], (np.integer,)) else float(top[k]) if isinstance(top[k], np.floating) else top[k]) for k in g.keys()}
        res = lb.run_backtest_bars(bars, key, params)
        results[key] = res
        print(f'{key:22} best={params} n={res["metrics"]["trades"]:4d} avgR={res["metrics"]["avg_r"]:.3f}')
    except Exception as e:
        print(f'{key}: error → {e}')

lb.export_best_params(results, 'best_params.json')
print('\n✓ escrito best_params.json — arrástralo al dashboard.')

## 7. (Opcional) Features para ML

Cada trade guarda un vector `features` con los mismos slots que el motor TS.
Puedes usarlo directo con sklearn/XGBoost para entrenar un clasificador que
filtre trades malos antes de enviarlos al EA.

In [ ]:
df_all = lb.trades_to_df(res['trades'])
X_cols = [c for c in df_all.columns if c.startswith('f_')]
y = (df_all['r'] > 0).astype(int)  # target: trade ganador
print('Features:', X_cols)
print('Balance clases:', y.value_counts().to_dict())